# ChEMBL Data Preprocessing → Interaction Fingerprints (IFP)

End-to-end pipeline for preparing ligand–receptor interaction data for DRD2.

**Steps:**
1. ChEMBL data filtering (deduplication, activity labeling)
2. Diverse active compound selection (MaxMin picking)
3. SDF preparation for docking
4. Best pose selection per ligand
5. Interaction fingerprint (IFP) generation

## Configuration
All file paths and activity thresholds

In [3]:
# Input files
CHEMBL_CSV   = "chembl.csv"        # ChEMBL website export (semicolon-separated)
DOCKED_SDF   = "D22.sdf"           # Docking output poses (Glide SP)
RECEPTOR_PDB = "D22.pdb"          # Receptor structure

# Activity thresholds
ACTIVE_THRESHOLD_NM   = 100        # Ki <= 100 nM   → active   (label=1)
INACTIVE_THRESHOLD_NM = 10_000     # Ki >= 10000 nM → inactive (label=0)
# Grey zone 100–10000 nM is excluded from the dataset

# Diverse subset selection
N_ACTIVES_SELECT = 804             # number of actives to pick (MaxMin)
MORGAN_RADIUS    = 2               # ECFP4
MORGAN_NBITS     = 2048
RANDOM_SEED      = 42

# IFP
LABEL_FIELD = "label"
DS_FIELD    = "r_i_docking_score"  # Glide SP docking score field in SDF

INTERACTIONS = [
    'Anionic', 'CationPi', 'Cationic',
    'HBAcceptor', 'HBDonor', 'Hydrophobic',
    'MetalAcceptor', 'MetalDonor', 'PiCation',
    'PiStacking', 'VdWContact', 'XBAcceptor', 'XBDonor'
]


## 1. ChEMBL Data Filtering

Load the ChEMBL CSV export and apply the following filters:
- Keep only Ki measurements (relations `=` and `>`)
- Remove duplicate records within the same assay
- For compounds with multiple measurements across assays:
  - all consistent → keep the record with the lowest Ki
  - conflicting labels or all in grey zone → discard compound
- Assign activity labels: `label=1` (active), `label=0` (inactive)
- Censored measurements `Ki > 10000 nM` are treated as inactive

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

MIN_HEAVY_ATOMS = 10


def assign_label(val, rel):
    if rel == "=" and val <= ACTIVE_THRESHOLD_NM:
        return 1
    if rel in ("=", ">", ">=") and val >= INACTIVE_THRESHOLD_NM:
        return 0
    return None

def resolve_duplicates(group):
    if len(group) == 1:
        return group
    labels = group.apply(
        lambda r: assign_label(r["Standard Value"], r["Standard Relation"]), axis=1
    )
    valid = labels.dropna()
    if len(valid) == 0 or valid.nunique() > 1:
        return None
    return group.loc[[group["Standard Value"].idxmin()]]

def clean_chembl(csv_path):
    raw = pd.read_csv(csv_path, sep=";")
    print(f"Raw records: {len(raw)}")

    df = raw.copy()
    df["Standard Value"] = pd.to_numeric(df["Standard Value"], errors="coerce")
    df = df.dropna(subset=["Smiles", "Standard Value"])
    df = df[df["Standard Value"] > 0]

    # Keep only Ki measurements
    df = df[df["Standard Type"] == "Ki"]

    # Standardise relation strings (remove quotes if present)
    df["Standard Relation"] = (
        df["Standard Relation"].astype(str).str.replace("'", "").str.strip()
    )
    df = df[df["Standard Relation"].isin(["=", ">", ">="])]

    # Drop duplicates within the same assay
    df = df.drop_duplicates(subset=["Molecule ChEMBL ID", "Assay ChEMBL ID"])

    # Resolve duplicates across assays
    n_before = df["Molecule ChEMBL ID"].nunique()
    parts = []
    for _, group in df.groupby("Molecule ChEMBL ID"):
        result = resolve_duplicates(group)
        if result is not None:
            parts.append(result)
    df = pd.concat(parts).reset_index(drop=True) if parts else df.iloc[0:0]
    n_after = df["Molecule ChEMBL ID"].nunique()
    print(f"Unique compounds: {n_before} → {n_after} "
          f"(removed {n_before - n_after} conflicting / grey-zone)")

    # Assign labels
    df["label"] = df.apply(
        lambda r: assign_label(r["Standard Value"], r["Standard Relation"]), axis=1
    )
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)

    # Column order
    front = ["Molecule ChEMBL ID", "Smiles", "label",
             "Standard Type", "Standard Value", "Standard Relation",
             "pChEMBL Value", "Assay ChEMBL ID", "Document ChEMBL ID"]
    front = [c for c in front if c in df.columns]
    df = df[front + [c for c in df.columns if c not in front]].reset_index(drop=True)

    active   = (df["label"] == 1).sum()
    inactive = (df["label"] == 0).sum()
    print(f"After filtering: {len(df)} compounds  |  "
          f"active: {active}  |  inactive: {inactive}")
    print(f"Active/inactive ratio: {active / max(inactive, 1):.2f}")
    return df


df_clean = clean_chembl(CHEMBL_CSV)
df_clean.to_csv("chembl_clean.csv", index=False)
df_clean.head(3)

Raw records: 14791
Unique compounds: 9735 → 8582 (removed 1153 conflicting / grey-zone)
After filtering: 4029 compounds  |  active: 3225  |  inactive: 804
Active/inactive ratio: 4.01


,Molecule ChEMBL ID,Smiles,label,Standard Type,Standard Value,Standard Relation,pChEMBL Value,Assay ChEMBL ID,Document ChEMBL ID,Molecule Name,...,Target Type,Source ID,Source Description,Document Journal,Document Year,Cell ChEMBL ID,Properties,Action Type,Standard Text Value,Value
0,CHEMBL100454,CN1CCN(C2=Nc3ccccc3Nc3sc(CO)cc32)CC1,1,Ki,22.0,=,7.66,CHEMBL671419,CHEMBL1150510,NaN,...,SINGLE PROTEIN,1,Scientific Literature,Bioorg Med Chem Lett,1997.0,NaN,NaN,NaN,NaN,22.0
1,CHEMBL100461,Oc1ccc(OCCNCc2ccccc2)cc1,0,Ki,10000.0,>,NaN,CHEMBL671654,CHEMBL1131356,NaN,...,SINGLE PROTEIN,1,Scientific Literature,Bioorg Med Chem Lett,1998.0,CHEMBL3308072,NaN,NaN,NaN,10000.0
2,CHEMBL10085,CC(C)Oc1ccccc1N1CCN(Cc2cccc(C(=O)N3CCCCC3)c2)CC1,1,Ki,2.2,=,8.66,CHEMBL670744,CHEMBL1129884,MAZAPERTINE,...,SINGLE PROTEIN,1,Scientific Literature,Bioorg Med Chem Lett,1997.0,NaN,NaN,NaN,NaN,2.2


---
## 2. Diverse Active Compound Selection

Balance the dataset by selecting `N_ACTIVES_SELECT` actives using **MaxMin picking** on Morgan fingerprints (ECFP4). MaxMin iteratively picks the compound most dissimilar to all already-selected ones (Tanimoto distance), maximising chemical space coverage.

In [ ]:
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.SimDivFilters import rdSimDivPickers


def smiles_to_fp(smi):
    """Convert SMILES to Morgan fingerprint"""
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    gen = AllChem.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS)
    return gen.GetFingerprint(mol)


def maxmin_select(df_actives, n, seed):
    """Run MaxMin picking and return a DataFrame of selected compounds."""
    fps, valid_idx = [], []
    for i, smi in enumerate(df_actives["Smiles"]):
        fp = smiles_to_fp(smi)
        if fp is not None:
            fps.append(fp)
            valid_idx.append(i)

    df_valid = df_actives.iloc[valid_idx].reset_index(drop=True)
    n = min(n, len(fps))
    print(f"MaxMin picking: selecting {n} from {len(fps)} actives...")

    picker  = rdSimDivPickers.MaxMinPicker()
    dist_fn = lambda i, j: 1.0 - DataStructs.TanimotoSimilarity(fps[i], fps[j])
    picks   = list(picker.LazyPick(dist_fn, len(fps), n, firstPicks=[seed % len(fps)]))

    selected = df_valid.iloc[picks].reset_index(drop=True)

    # Diversity statistics on a random sample of 500 pairs
    import random; random.seed(seed)
    sample = random.sample(range(len(picks)), min(500, len(picks)))
    sims = [
        DataStructs.TanimotoSimilarity(fps[picks[i]], fps[picks[j]])
        for i in range(len(sample)) for j in range(i + 1, len(sample))
    ]
    print(f"Mean Tanimoto similarity of selected set: {np.mean(sims):.3f} "
          f"(lower = more diverse)")
    return selected


actives   = df_clean[df_clean["label"] == 1].copy().reset_index(drop=True)
inactives = df_clean[df_clean["label"] == 0].copy().reset_index(drop=True)

selected_actives = maxmin_select(actives, N_ACTIVES_SELECT, RANDOM_SEED)

df_balanced = pd.concat([selected_actives, inactives], ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"\nFinal dataset: {len(df_balanced)} compounds  |  "
      f"active: {(df_balanced['label']==1).sum()}  |  "
      f"inactive: {(df_balanced['label']==0).sum()}")

df_balanced.to_csv("chembl_balanced.csv", index=False)
df_balanced["label"].value_counts()


MaxMin picking: selecting 804 from 3225 actives...
Mean Tanimoto similarity of selected set: 0.154 (lower = more diverse)

Final dataset: 1608 compounds  |  active: 804  |  inactive: 804


label
1    804
0    804
Name: count, dtype: int64

## 3. SDF Preparation for Docking

Convert SMILES to SDF with 2D coordinates. 

In [5]:
from rdkit.Chem import AllChem
from rdkit.Chem.rdmolops import RemoveHs


def smiles_to_mol_2d(smi):
    """Parse SMILES and generate 2D coordinates; returns None for invalid SMILES."""
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    mol = RemoveHs(mol)           # remove explicit Hs – LigPrep will add its own
    AllChem.Compute2DCoords(mol)  # 2D coords required for valid SDF
    return mol


def write_sdf_for_docking(df, output_path):
    writer = Chem.SDWriter(output_path)
    ok, failed = 0, 0
    for _, row in df.iterrows():
        mol = smiles_to_mol_2d(row["Smiles"])
        if mol is None:
            failed += 1
            continue
        mol.SetProp("_Name",             str(row.get("Molecule ChEMBL ID", "")))
        mol.SetProp("label",             str(int(row["label"])))
        mol.SetProp("Standard Value",    str(row.get("Standard Value", "")))
        mol.SetProp("Standard Relation", str(row.get("Standard Relation", "")))
        writer.write(mol)
        ok += 1
    writer.close()
    print(f"Saved {ok} molecules to {output_path}  (skipped {failed} invalid SMILES)")


write_sdf_for_docking(df_balanced, "ligands_for_docking.sdf")


Saved 1608 molecules to ligands_for_docking.sdf  (skipped 0 invalid SMILES)


## 4. Best Pose Selection

After Glide SP docking each ligand may have multiple poses. We select the pose with the lowest `r_i_docking_score`.

We also compare compound lists before and after docking to identify which molecules failed to dock and what their activity labels are.

In [ ]:
from rdkit import Chem
import numpy as np


def select_best_poses(sdf_path, ds_field="r_i_docking_score"):
    """For each ligand, keep the pose with the lowest docking score."""
    suppl = Chem.SDMolSupplier(sdf_path, removeHs=False)
    best  = {}  # name → mol
    for mol in suppl:
        if mol is None or not mol.HasProp(ds_field):
            continue
        name  = mol.GetProp("_Name")
        score = float(mol.GetProp(ds_field))
        if name not in best or score < float(best[name].GetProp(ds_field)):
            best[name] = mol
    return list(best.values())


def compare_before_after(df_before, sdf_after):
    """Report which compounds did not dock and their label distribution."""
    suppl      = Chem.SDMolSupplier(sdf_after, removeHs=False)
    docked_ids = {m.GetProp("_Name") for m in suppl if m is not None}
    before_ids = set(df_before["Molecule ChEMBL ID"].astype(str))
    missing    = before_ids - docked_ids

    print(f"Before docking: {len(before_ids)}")
    print(f"After docking:  {len(docked_ids)}  ({len(docked_ids)/len(before_ids)*100:.1f}%)")
    print(f"Did not dock:   {len(missing)}  ({len(missing)/len(before_ids)*100:.1f}%)")

    if missing:
        miss_df = df_before[df_before["Molecule ChEMBL ID"].astype(str).isin(missing)]
        print("\nLabel distribution among non-docked compounds:")
        print(miss_df["label"].value_counts()
              .rename({1: "active", 0: "inactive"}).to_string())
        print("\nExample missing active compounds:")
        print(miss_df[miss_df["label"] == 1].head())
    return missing


# Compare before / after
missing_ids = compare_before_after(df_balanced, DOCKED_SDF)

# Select best pose per ligand
best_mols = select_best_poses(DOCKED_SDF)
print(f"\nBest poses selected: {len(best_mols)}")

# Save
out_path = "best_poses_DRD2.sdf"
writer = Chem.SDWriter(out_path)
for mol in best_mols:
    writer.write(mol)
writer.close()
print(f"Saved: {out_path}")

# Score statistics
scores = [float(m.GetProp(DS_FIELD)) for m in best_mols if m.HasProp(DS_FIELD)]
print(f"Docking score: min={min(scores):.2f}, max={max(scores):.2f}, "
      f"mean={np.mean(scores):.2f}")


Before docking: 1608
After docking:  1583  (98.4%)
Did not dock:   25  (1.6%)

Label distribution among non-docked compounds:
label
active      19
inactive     6

Best poses selected: 1583
Saved: best_poses_DRD2.sdf
Docking score: min=-11.02, max=-3.43, mean=-7.31


## 5. Interaction Fingerprint (IFP) Generation

ProLIF computes binary protein–ligand interaction fingerprints for each pose.

**Interaction types:** Anionic, CationPi, Cationic, HBAcceptor, HBDonor, Hydrophobic, MetalAcceptor, MetalDonor, PiCation, PiStacking, VdWContact, XBAcceptor, XBDonor

Output columns: `molecule_name`, `label`, + IFP bits named `{RESNAME}{RESSEQ}{CHAIN}_{InteractionType}`.

In [7]:
import prolif
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, MolFromPDBFile


def fix_mol(mol):
    """Fix aromaticity and add explicit hydrogens with coordinates."""
    mol = Chem.RWMol(mol)
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        Chem.SanitizeMol(mol, Chem.SanitizeFlags.SANITIZE_ALL ^
                              Chem.SanitizeFlags.SANITIZE_PROPERTIES)
    mol = AllChem.AddHs(mol, addCoords=True)
    return mol


def compute_fingerprints(sdf_path, receptor_pdb, interactions, label_field="label"):
    """
    Compute binary IFP for every ligand pose in sdf_path against receptor_pdb.

    Returns a DataFrame with columns:
      molecule_name | label | {RESNAME}{RESSEQ}{CHAIN}_{InteractionType} ...
    """
    # Load receptor
    print(f"Loading receptor: {receptor_pdb}")
    prot_rd = MolFromPDBFile(receptor_pdb, removeHs=False, sanitize=False)
    AllChem.SanitizeMol(
        prot_rd,
        Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_PROPERTIES
    )
    prot_plif = prolif.Molecule(prot_rd)
    print(f"Receptor: {prot_rd.GetNumAtoms()} atoms, "
          f"{sum(1 for a in prot_rd.GetAtoms() if a.GetIsAromatic())} aromatic")

    # Load ligands – two parallel readers to keep name/label aligned with IFP rows
    print(f"Loading ligands from {sdf_path}")
    suppl_rd  = Chem.SDMolSupplier(sdf_path, removeHs=False, sanitize=True)
    lig_suppl = prolif.sdf_supplier(sdf_path, sanitize=True)

    records  = []  # [{molecule_name, label}] – grows in lockstep with lig_list
    lig_list = []

    for mol_rd, mol_plif in zip(suppl_rd, lig_suppl):
        if mol_rd is None:
            continue
        name  = mol_rd.GetProp("_Name") if mol_rd.HasProp("_Name") else None
        label = mol_rd.GetProp(label_field) if mol_rd.HasProp(label_field) else None
        fixed = fix_mol(mol_rd)
        lig_list.append(prolif.Molecule(fixed))
        records.append({
            "molecule_name": name,
            "label": int(label) if label is not None else None,
        })

    print(f"Computing IFP for {len(lig_list)} ligands...")
    fp = prolif.Fingerprint(interactions)
    fp.run_from_iterable(lig_list, prot_plif, progress=True)
    df_fp = fp.to_dataframe()

    # Flatten MultiIndex columns (ProLIF 2.x: 3-level MultiIndex)
    new_cols = []
    for col in df_fp.columns:
        if col == "Frame":
            new_cols.append("Frame")
        else:
            # e.g. ('LIG1', 'ASP115A', 'HBAcceptor') → 'ASP115A_HBAcceptor'
            parts = [str(p) for p in col if str(p) not in ("", "UNK")]
            new_cols.append("_".join(parts[-2:]))
    df_fp.columns = new_cols
    df_fp = df_fp.loc[:, ~df_fp.columns.duplicated()]
    df_fp = df_fp.astype(int).reset_index(drop=True)

    # Attach metadata – same index guarantees alignment with IFP rows
    meta_df = pd.DataFrame(records).reset_index(drop=True)
    result  = pd.concat([meta_df, df_fp], axis=1)

    print(f"Done. Shape: {result.shape}")
    return result


df_ifp = compute_fingerprints("best_poses_DRD2.sdf", RECEPTOR_PDB, INTERACTIONS)
df_ifp.to_csv("ifp_DRD2.csv", index=False)
df_ifp[["molecule_name", "label"]].head(5)


c:\Users\kinga\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\MDAnalysis\topology\tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


Loading receptor: D22.pdb
Receptor: 6876 atoms, 331 aromatic
Loading ligands from best_poses_DRD2.sdf
Computing IFP for 1583 ligands...


100%|██████████| 1583/1583 [00:25<00:00, 61.25it/s] 


Done. Shape: (1583, 118)


,molecule_name,label
0,CHEMBL83617,1
1,CHEMBL3102875,1
2,CHEMBL4750660,0
3,CHEMBL4751063,1
4,CHEMBL3423332,1


### Order sanity check

In [8]:
# Verify that molecule_name in the DataFrame matches _Name fields in the SDF
suppl_check = Chem.SDMolSupplier("best_poses_DRD2.sdf", sanitize=True)
names_sdf   = [m.GetProp("_Name") for m in suppl_check if m is not None]
names_df    = df_ifp["molecule_name"].tolist()

assert names_sdf == names_df, "ORDER MISMATCH – check the loading loop!"
print(f"OK – order verified ({len(names_df)} ligands)")

print(f"\nIFP summary:")
print(f"  Shape:          {df_ifp.shape}")
print(f"  Active:         {(df_ifp['label']==1).sum()}")
print(f"  Inactive:       {(df_ifp['label']==0).sum()}")
print(f"  IFP bits:       {df_ifp.shape[1] - 2}")
print(f"  Always-zero:    {(df_ifp.drop(columns=['molecule_name','label'])==0).all().sum()}")


OK – order verified (1583 ligands)

IFP summary:
  Shape:          (1583, 118)
  Active:         785
  Inactive:       798
  IFP bits:       116
  Always-zero:    0
